# Frequency Branch — NFI Deepfake Dataset

Trains and tests the **frequency-domain branch** (FFT/DCT artifact
detector) of the team's unified fusion pipeline, on the NFI rPPG
Deepfake Detection dataset.

Output: `frequency_test_preds.json`, matching the exact schema
`fusion/contract.py` expects:
```json
{"branch": "frequency", "predictions": [{"video_id": "...", "score": 0.XX}, ...]}
```

This notebook only scores the videos listed in
`data/manifests/frequency_manifest.json` (train/val to fit the model,
test to produce the final submission) — not necessarily the full dataset.

**Dataset source:** the NFI dataset is no longer available on Hugging
Face. This version reads the videos from a zip you've uploaded to your
own Google Drive instead (`real/`, `fake/`, `annotations.json` at the
top level — same layout the original HF dataset used).

Run top to bottom. GPU runtime recommended (Runtime → Change runtime
type → T4 GPU), though this branch is classical-ML (CPU-only) so it
will still work without GPU, just slower.

## Stage 0 — Setup

**Before running this notebook:**
1. Upload your dataset zip (e.g. `raw_videos-....zip`) to your Google
   Drive — anywhere is fine, e.g. `My Drive/nfi_dataset/raw_videos.zip`.
2. Set `DRIVE_ZIP_PATH` below to that exact path.

You also need `data/manifests/frequency_manifest.json` from your team's
repo — this cell clones it automatically if the repo is reachable,
otherwise it'll prompt you to upload the file manually.

In [11]:
import os
!pip -q install opencv-python-headless scikit-learn scipy tqdm pandas numpy

In [12]:
from google.colab import drive
drive.mount("/content/drive")

# <<< EDIT THIS to match where you put the zip in your Drive >>>
DRIVE_ZIP_PATH = "/content/drive/MyDrive/raw_videos-20261029T190810Z-1-001.zip"

if not os.path.exists(DRIVE_ZIP_PATH):
    print(f"⚠️ Not found at {DRIVE_ZIP_PATH}")
    print("   Either fix DRIVE_ZIP_PATH above to the real location, or upload here:")
    from google.colab import files
    up = files.upload()
    DRIVE_ZIP_PATH = "/content/" + list(up)[0]
print("zip:", DRIVE_ZIP_PATH, round(os.path.getsize(DRIVE_ZIP_PATH) / 1e9, 2), "GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
zip: /content/drive/MyDrive/raw_videos-20261029T190810Z-1-001.zip 1.89 GB


In [13]:
import os
import json

REPO_URL = "https://github.com/kalingasajja/FAS-deepfake-capstone.git"
REPO_DIR = "/content/FAS-deepfake-capstone"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

MANIFEST_PATH = os.path.join(
    REPO_DIR, "data", "manifests", "frequency_manifest.json"
)

if not os.path.isfile(MANIFEST_PATH):
    print("Manifest not found in repository.")
    print("Please upload frequency_manifest.json")

    from google.colab import files
    uploaded = files.upload()

    if not uploaded:
        raise FileNotFoundError("Manifest upload was cancelled.")

    MANIFEST_PATH = os.path.abspath(next(iter(uploaded)))

with open(MANIFEST_PATH, "r") as f:
    MANIFEST = json.load(f)

print("Manifest:", MANIFEST_PATH)
print("Branch:", MANIFEST.get("branch"))

for split in ["train", "val", "test"]:
    print(split, ":", len(MANIFEST[split]), "videos")

Cloning into '/content/FAS-deepfake-capstone'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 125 (delta 37), reused 101 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 5.79 MiB | 21.96 MiB/s, done.
Resolving deltas: 100% (37/37), done.
Manifest: /content/FAS-deepfake-capstone/data/manifests/frequency_manifest.json
Branch: frequency
train : 346 videos
val : 87 videos
test : 67 videos


### Config

In [14]:
import os, sys, json, time, random, shutil, glob, math, warnings
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
cv2.setNumThreads(2)

CFG = dict(
    WORK="/content/work",
    DATASET_DIR="/content/nfi_dataset/raw_videos",
    T=8,
    SIZE=224,
    CLIPS_PER_VIDEO=2,
    SEED=42,
)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])

def P(*a): return os.path.join(CFG["WORK"], *a)
for d in [P("clips", "clean"), P("models"), "results"]:
    os.makedirs(d, exist_ok=True)

MANIFEST = json.load(open(MANIFEST_PATH))
print("branch:", MANIFEST.get("branch"))
for split in ["train", "val", "test"]:
    print(split, ":", len(MANIFEST[split]), "videos")

branch: frequency
train : 346 videos
val : 87 videos
test : 67 videos


## Stage 1 — Extract the dataset zip and resolve each manifest video's local path

Extracts once to local Colab disk (`/content/nfi_dataset`) — much faster
to read videos from than directly out of Drive. Re-running this cell on
the same session skips re-extraction if it already happened.

In [15]:
if not os.path.exists(os.path.join(CFG["DATASET_DIR"], "fake")) and \
   not os.path.exists(os.path.join(CFG["DATASET_DIR"], "real")):
    os.makedirs(CFG["DATASET_DIR"], exist_ok=True)
    print("extracting zip (this can take a few minutes for ~1.9 GB)...")
    import zipfile
    with zipfile.ZipFile(DRIVE_ZIP_PATH) as z:
        z.extractall(CFG["DATASET_DIR"])
    print("✅ extracted to", CFG["DATASET_DIR"])
else:
    print("already extracted, skipping:", CFG["DATASET_DIR"])

!find {CFG["DATASET_DIR"]} -maxdepth 2 -type d
print("---")
!find {CFG["DATASET_DIR"]} -iname "*.mp4" | head -5

already extracted, skipping: /content/nfi_dataset/raw_videos
/content/nfi_dataset/raw_videos
/content/nfi_dataset/raw_videos/real
/content/nfi_dataset/raw_videos/real/CelebDF
/content/nfi_dataset/raw_videos/.cache
/content/nfi_dataset/raw_videos/.cache/huggingface
/content/nfi_dataset/raw_videos/fake
/content/nfi_dataset/raw_videos/fake/Kaggle_FS
/content/nfi_dataset/raw_videos/fake/PixVerse_SYN
/content/nfi_dataset/raw_videos/fake/Deepfaker_FS
/content/nfi_dataset/raw_videos/fake/AkoolAI_FS
/content/nfi_dataset/raw_videos/fake/Pika_SYN
/content/nfi_dataset/raw_videos/fake/Luma_SYN
/content/nfi_dataset/raw_videos/fake/KlingAI_SYN
/content/nfi_dataset/raw_videos/fake/Runway_SYN
/content/nfi_dataset/raw_videos/fake/RefaceAI_FS
/content/nfi_dataset/raw_videos/fake/Faceswapper_FS
/content/nfi_dataset/raw_videos/fake/Sora_SYN
---
/content/nfi_dataset/raw_videos/real/CelebDF/id8_0000.mp4
/content/nfi_dataset/raw_videos/real/CelebDF/id1_0008.mp4
/content/nfi_dataset/raw_videos/real/CelebDF/id

In [16]:
def resolve_local_paths(manifest, dataset_dir):
    all_entries = manifest["train"] + manifest["val"] + manifest["test"]
    local_paths = {}
    missing = []
    for e in all_entries:
        candidate = os.path.join(dataset_dir, e["path"])
        if os.path.exists(candidate):
            local_paths[e["video_id"]] = candidate
        else:
            missing.append((e["video_id"], e["path"]))
    if missing:
        print(f"⚠️ {len(missing)} manifest videos not found under {dataset_dir}, e.g.:")
        for vid, p in missing[:8]:
            print(" ", vid, "->", p)
        print("   If this list is large, the zip's internal folder layout doesn't match")
        print("   the manifest's 'path' field — paste one of the lines above plus the")
        print("   actual folder tree (the 'find ... -maxdepth 2' output) so the path")
        print("   resolution logic can be adjusted.")
    print(f"✅ resolved {len(local_paths)}/{len(all_entries)} videos")
    return local_paths

LOCAL_PATHS = resolve_local_paths(MANIFEST, CFG["DATASET_DIR"])

✅ resolved 500/500 videos


## Stage 2 — Face-crop clips (8 consecutive frames x 2 clips per video)

Same face-detect-and-crop logic as the earlier deepfake/SiW notebooks.

In [17]:
!pip -q install facenet-pytorch --no-deps

In [18]:
class FaceDetector:
    """MTCNN only. (Some Colab cv2 builds ship without CascadeClassifier /
    objdetect, so there is no Haar fallback here — if MTCNN fails to
    install, fix that instead of silently degrading detection quality.)"""
    def __init__(self):
        import torch
        from facenet_pytorch import MTCNN
        self.mtcnn = MTCNN(keep_all=True, device="cuda" if torch.cuda.is_available() else "cpu", post_process=False)
        print("face detector: MTCNN")
        self.stats = {"mtcnn": 0, "none": 0}

    def _box(self, x1, y1, x2, y2, s):
        w, h = (x2 - x1) / s, (y2 - y1) / s
        return ((x1 + x2) / 2 / s, (y1 + y2) / 2 / s, max(w, h) * 1.3)

    def detect(self, bgr):
        H, W = bgr.shape[:2]
        s = min(1.0, 640.0 / max(H, W))
        small = cv2.resize(bgr, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else bgr
        try:
            boxes, _ = self.mtcnn.detect(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
            if boxes is not None and len(boxes):
                b = max(boxes, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
                cx, cy, side = self._box(*b, 1.0)
                self.stats["mtcnn"] += 1
                return (cx / s, cy / s, side / s)
        except Exception:
            pass
        self.stats["none"] += 1
        return None

def read_clip(path, start, T):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(T):
        ok, f = cap.read()
        if not ok: break
        frames.append(f)
    cap.release()
    return frames

def crop_resize(bgr, box, size):
    cx, cy, side = box
    s = int(round(side)); x1 = int(round(cx - side / 2)); y1 = int(round(cy - side / 2))
    x2, y2 = x1 + s, y1 + s
    H, W = bgr.shape[:2]
    pl, pt, pr, pb = max(0, -x1), max(0, -y1), max(0, x2 - W), max(0, y2 - H)
    if pl or pt or pr or pb:
        bgr = cv2.copyMakeBorder(bgr, pt, pb, pl, pr, cv2.BORDER_REPLICATE)
        x1 += pl; x2 += pl; y1 += pt; y2 += pt
    crop = bgr[y1:y2, x1:x2]
    interp = cv2.INTER_AREA if s > size else cv2.INTER_CUBIC
    return cv2.resize(crop, (size, size), interpolation=interp)

def process_video(path, video_id, split, label, det):
    T, S = CFG["T"], CFG["SIZE"]
    cap = cv2.VideoCapture(path); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    if n < T + 2: return []
    outdir = P("clips", "clean"); os.makedirs(outdir, exist_ok=True)
    rows = []
    for k, frac in enumerate((0.2, 0.6)[: CFG["CLIPS_PER_VIDEO"]]):
        frames = read_clip(path, int(frac * (n - T)), T)
        if len(frames) < T: continue
        box = det.detect(frames[T // 2])
        if box is None: continue
        crops = np.stack([crop_resize(f, box, S) for f in frames])[..., ::-1]   # BGR -> RGB
        cid = f"{video_id}__c{k}"
        np.save(f"{outdir}/{cid}.npy", np.ascontiguousarray(crops))
        rows.append(dict(clip_id=cid, video_id=video_id, label=int(label), split=split, variant="clean"))
    return rows

In [19]:
META_CSV = P("meta_clean.csv")
if not os.path.exists(META_CSV):
    det = FaceDetector()
    rows = []
    for split in ["train", "val", "test"]:
        for e in tqdm(MANIFEST[split], desc=f"clips [{split}]"):
            local = LOCAL_PATHS.get(e["video_id"])
            if not local or not os.path.exists(local):
                continue
            rows += process_video(local, e["video_id"], split, e["label"], det)
    print("detector usage:", det.stats)
    META = pd.DataFrame(rows)
    META.to_csv(META_CSV, index=False)
else:
    META = pd.read_csv(META_CSV)

print(META.groupby("split").size())
print("videos with >=1 clip:", META.video_id.nunique(), "/ manifest total:",
      sum(len(MANIFEST[s]) for s in ["train", "val", "test"]))
META_ALL = META.copy()

face detector: MTCNN


clips [train]:   0%|          | 0/346 [00:00<?, ?it/s]

clips [val]:   0%|          | 0/87 [00:00<?, ?it/s]

clips [test]:   0%|          | 0/67 [00:00<?, ?it/s]

detector usage: {'mtcnn': 999, 'none': 1}
split
test     134
train    691
val      174
dtype: int64
videos with >=1 clip: 500 / manifest total: 500


## Stage 3 — Frequency-domain features (F group)

FFT/DCT radial power spectrum, spectral slope, high-frequency energy,
spectral-peak anomalies, DCT block-energy coefficients, JPEG blockiness,
chroma-channel spectra, and row/column banding — these catch GAN
upsampling artifacts and recompression/replay artifacts invisible in the
raw pixel domain. This is the full feature set for the frequency branch
(no boundary/temporal features — those belong to other branches).

In [20]:
import scipy.fft
S_ = CFG["SIZE"]; NB = 32
_yy, _xx = np.indices((S_, S_)); _R = np.hypot(_yy - (S_ - 1) / 2, _xx - (S_ - 1) / 2)
_RB = np.minimum((_R / (S_ / 2) * NB).astype(int), NB)                 # radial bin (NB == outside)
_HAN = np.outer(np.hanning(S_), np.hanning(S_)).astype(np.float32)

def _radial(P2, nb, rb):
    m = rb < nb
    return np.bincount(rb[m], P2[m], minlength=nb) / np.maximum(np.bincount(rb[m], minlength=nb), 1)

def _logspec(ch):
    F = np.fft.fftshift(np.fft.fft2((ch - ch.mean()) * _HAN))
    Pw = np.abs(F) ** 2
    return np.log10(Pw + 1e-8), Pw

def _band_peak(v):
    v = v - np.polyval(np.polyfit(np.arange(len(v)), v, 2), np.arange(len(v)))
    s = np.abs(np.fft.rfft(v * np.hanning(len(v))))[2:60]
    return float(s.max() / (s.mean() + 1e-6))

def _dct_feats(ch):
    b = ch.reshape(28, 8, 28, 8).transpose(0, 2, 1, 3).reshape(-1, 8, 8)
    d = scipy.fft.dctn(b - 128.0, axes=(1, 2), norm="ortho")
    L = np.log1p(np.abs(d)).mean(0); s = np.indices((8, 8)).sum(0)
    return [L[s == k].mean() for k in range(1, 15)], float((np.abs(d[:, 4:, 4:]) < 0.5).mean())

def frame_feats(rgb):
    f = {}
    ycc = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb).astype(np.float32)
    Y, Cr, Cb = ycc[..., 0], ycc[..., 1], ycc[..., 2]
    P2, Pw = _logspec(Y)
    prof = _radial(P2, NB, _RB)
    for i, v in enumerate(prof): f[f"f_rad{i:02d}"] = v
    f["f_slope"] = np.polyfit(np.log(np.arange(4, NB) + 0.5), prof[4:], 1)[0]
    inside = _R < S_ / 2
    f["f_hf"] = np.log10(Pw[inside & (_R > 0.5 * S_ / 2)].sum() / Pw[inside].sum() + 1e-12)
    exp2d = prof[np.minimum(_RB, NB - 1)]
    msk = (_R >= 0.12 * S_ / 2) & inside
    r = (P2 - exp2d)[msk]; rr = _R[msk]
    f["f_pk_max"] = r.max(); f["f_pk_p999"] = np.percentile(r, 99.9)
    f["f_pk_frac"] = float((r > 3 * r.std()).mean()); f["f_pk_rad"] = rr[r.argmax()] / (S_ / 2)
    dY, zr = _dct_feats(Y)
    for i, v in enumerate(dY): f[f"f_dct{i + 1:02d}"] = v
    f["f_dct_zero"] = zr
    dxx = np.abs(np.diff(Y, axis=1)); dyy = np.abs(np.diff(Y, axis=0))
    f["f_blk_h"] = dxx[:, 7::8].mean() / (dxx.mean() + 1e-6); f["f_blk_v"] = dyy[7::8, :].mean() / (dyy.mean() + 1e-6)
    for nm, ch in (("cr", Cr), ("cb", Cb)):
        pc = _radial(_logspec(ch)[0], 8, np.minimum((_R / (S_ / 2) * 8).astype(int), 8))
        for i, v in enumerate(pc): f[f"f_{nm}_rad{i}"] = v
    f["f_band_row"] = _band_peak(Y.mean(1)); f["f_band_col"] = _band_peak(Y.mean(0))
    return f

def clip_features(clip):
    clip = np.asarray(clip)
    T = len(clip); idx = [0, T // 2, T - 1]
    fr = [frame_feats(np.ascontiguousarray(clip[i])) for i in idx]
    return {k: float(np.mean([d[k] for d in fr])) for k in fr[0]}

def _feat_job(cid, path):
    import cv2 as _cv2; _cv2.setNumThreads(1)
    d = clip_features(np.load(path)); d["clip_id"] = cid
    return d

def compute_features(meta):
    from joblib import Parallel, delayed
    jobs = [(r.clip_id, P("clips", "clean", r.clip_id + ".npy")) for r in meta.itertuples()]
    out = Parallel(n_jobs=max(1, os.cpu_count() or 1), batch_size=8)(delayed(_feat_job)(*j) for j in tqdm(jobs, desc="features"))
    return meta.merge(pd.DataFrame(out), on="clip_id", how="left")

In [21]:
FEAT_CSV = P("features.csv")
if not os.path.exists(FEAT_CSV):
    FEAT = compute_features(META_ALL)
    FEAT.to_csv(FEAT_CSV, index=False)
else:
    FEAT = pd.read_csv(FEAT_CSV)
FCOLS = [c for c in FEAT.columns if c.startswith("f_")]
FEAT[FCOLS] = FEAT[FCOLS].replace([np.inf, -np.inf], np.nan)
print("features:", len(FCOLS), "| rows:", len(FEAT), "| NaNs:", int(FEAT[FCOLS].isna().sum().sum()))

features:   0%|          | 0/999 [00:00<?, ?it/s]

features: 73 | rows: 999 | NaNs: 0


## Stage 4 — Train classifier and score the test split

In [22]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

def train_hgb(df, cols):
    m = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06, max_leaf_nodes=15,
                                       l2_regularization=1.0, class_weight="balanced",
                                       random_state=CFG["SEED"])
    m.fit(df[cols].values, df.label.values)
    return m

tr = FEAT[FEAT.split == "train"]
va = FEAT[FEAT.split == "val"]
te = FEAT[FEAT.split == "test"]
print("train clips:", len(tr), "| val clips:", len(va), "| test clips:", len(te))

MODEL = train_hgb(tr, FCOLS)

val_prob = MODEL.predict_proba(va[FCOLS].values)[:, 1]
if va.label.nunique() > 1:
    print(f"Validation AUC: {roc_auc_score(va.label, val_prob):.3f}")

train clips: 691 | val clips: 174 | test clips: 134
Validation AUC: 0.973


## Stage 5 — Score test clips, aggregate to video-level, write submission JSON

The fusion contract expects ONE score per video_id (not per clip), so we
average clip-level probabilities within each video.

In [23]:
te = te.copy()
te["clip_prob"] = MODEL.predict_proba(te[FCOLS].values)[:, 1]
video_scores = te.groupby("video_id")["clip_prob"].mean().round(4)

if te.label.nunique() > 1:
    video_labels = te.groupby("video_id")["label"].first()
    auc = roc_auc_score(video_labels, video_scores.reindex(video_labels.index))
    acc = accuracy_score(video_labels, (video_scores.reindex(video_labels.index) >= 0.5).astype(int))
    print(f"Test video-level AUC: {auc:.3f} | ACC @ 0.5: {acc:.3f}")

# Make sure every manifest test video_id has a score, even ones that lost
# all their clips to a failed download or failed face-detection (fall back
# to 0.5 = "uncertain" so the fusion step doesn't silently drop the video).
test_ids = [e["video_id"] for e in MANIFEST["test"]]
missing = [vid for vid in test_ids if vid not in video_scores.index]
if missing:
    print(f"⚠️ {len(missing)} test videos have no usable clip (download/face-detect failure).")
    print("   Falling back to score=0.5 for these so they still appear in the submission:")
    for vid in missing[:10]:
        print("  ", vid)

predictions = []
for vid in test_ids:
    score = float(video_scores.get(vid, 0.5))
    predictions.append({"video_id": vid, "score": round(score, 4)})

submission = {"branch": "frequency", "predictions": predictions}
OUT_PATH = "results/frequency_test_preds.json"
os.makedirs("results", exist_ok=True)
with open(OUT_PATH, "w") as f:
    json.dump(submission, f, indent=2)
print(f"✅ wrote {len(predictions)} predictions to {OUT_PATH}")

Test video-level AUC: 0.982 | ACC @ 0.5: 0.955
✅ wrote 67 predictions to results/frequency_test_preds.json


## Stage 6 — Validate against the team's own contract checker

Uses `fusion/contract.py` from the cloned repo (same file your
teammates' fusion step runs) so a pass here means the real pipeline
will accept it too.

In [24]:
import sys
sys.path.insert(0, REPO_DIR)
from fusion.contract import validate_branch_file

records = validate_branch_file(OUT_PATH, expected_branch="frequency")

# Cross-check coverage against the manifest's test split specifically
manifest_test_ids = set(e["video_id"] for e in MANIFEST["test"])
pred_ids = set(r["video_id"] for r in records)
missing_from_submission = manifest_test_ids - pred_ids
extra_in_submission = pred_ids - manifest_test_ids
print(f"\nManifest test videos:        {len(manifest_test_ids)}")
print(f"Predictions submitted:       {len(pred_ids)}")
print(f"Missing from submission:     {len(missing_from_submission)}")
print(f"Unexpected extra video_ids:  {len(extra_in_submission)}")
if not missing_from_submission and not extra_in_submission:
    print("\n✅ PASS — exact match with manifest test set. Ready to submit.")
else:
    print("\n⚠️ Mismatch — check the lists above before submitting.")
    if missing_from_submission:
        print("Missing:", sorted(missing_from_submission)[:10])
    if extra_in_submission:
        print("Extra:", sorted(extra_in_submission)[:10])

[OK] Validated 67 entries in results/frequency_test_preds.json

Manifest test videos:        67
Predictions submitted:       67
Missing from submission:     0
Unexpected extra video_ids:  0

✅ PASS — exact match with manifest test set. Ready to submit.


In [25]:
# Download the final file to your laptop
from google.colab import files
files.download(OUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>